# Problem: Implement a CNN for CIFAR-10 (With Custom Layers)

### Problem Statement
You are tasked with implementing a **Convolutional Neural Network (CNN)** for image classification on the **CIFAR-10** dataset using PyTorch. However, instead of using PyTorch's built-in `nn.Conv2d` and `nn.MaxPool2d`, you must implement these layers **from scratch** using `nn.Module`. Your model will include convolutional layers for feature extraction, pooling layers for downsampling, and fully connected layers for classification.

### Requirements
1. **Implement Custom Layers**:
   - Create a custom `Conv2dCustom` class that mimics the behavior of `nn.Conv2d`.
   - Create a custom `MaxPool2dCustom` class that mimics the behavior of `nn.MaxPool2d`.

2. **Define the CNN Model**:
   - Use `Conv2dCustom` for convolutional layers.
   - Use `MaxPool2dCustom` for pooling layers.
   - Use standard `nn.Linear` for fully connected layers.
   - The model should process input images of shape `(3, 32, 32)` as in the CIFAR-10 dataset.

### Constraints
- You must not use `nn.Conv2d` or `nn.MaxPool2d`. Use your own custom implementations.
- The CNN should include multiple convolutional and pooling layers, followed by fully connected layers.
- Ensure the model outputs class predictions for **10 classes**, as required by CIFAR-10.

<details>
  <summary>💡 Hint</summary>
  Define `Conv2dCustom` and `MaxPool2dCustom` as subclasses of `nn.Module`, with the same weight layout as `nn.Conv2d`: `(out_channels, in_channels, kernel_h, kernel_w)`, plus one bias per output channel.  
  Nested loops over (batch, channel, row, column) with tensor slicing are the easiest way to get the arithmetic right — check them against `F.conv2d` / `F.max_pool2d` — but they are far too slow to train on CIFAR-10. Vectorize instead: `Tensor.unfold` (or `F.unfold`, a.k.a. im2col) extracts every sliding window at once, and the convolution becomes a single `einsum` / matmul over the window.  
  Output size per spatial dim: `(H + 2*padding - kernel) // stride + 1`.
</details>


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F

In [ ]:
# Load CIFAR-10 dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
def _pair(v):
    return v if isinstance(v, tuple) else (v, v)


class Conv2dCustom(nn.Module):
    """2D convolution (a cross-correlation, like nn.Conv2d) built from tensor ops."""

    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super(Conv2dCustom, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = _pair(kernel_size)
        self.stride = _pair(stride)
        self.padding = _pair(padding)

        # Same layout and default init as nn.Conv2d: weight (C_out, C_in, KH, KW), and
        # weight and bias drawn from U(-1/sqrt(fan_in), 1/sqrt(fan_in)).
        fan_in = in_channels * self.kernel_size[0] * self.kernel_size[1]
        bound = fan_in ** -0.5
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels, *self.kernel_size).uniform_(-bound, bound))
        self.bias = nn.Parameter(torch.empty(out_channels).uniform_(-bound, bound))

    def forward(self, x):
        KH, KW = self.kernel_size
        SH, SW = self.stride
        PH, PW = self.padding

        x = F.pad(x, (PW, PW, PH, PH))  # zero-pad left/right, then top/bottom
        # Every KH x KW window, as a strided view (no copy): (N, C_in, OH, OW, KH, KW)
        # with OH = (H + 2*PH - KH) // SH + 1 and OW = (W + 2*PW - KW) // SW + 1
        windows = x.unfold(2, KH, SH).unfold(3, KW, SW)
        # Dot every window with every filter, summing over (C_in, KH, KW): (N, C_out, OH, OW)
        out = torch.einsum("nchwij,ocij->nohw", windows, self.weight)
        return out + self.bias.view(1, -1, 1, 1)


class MaxPool2dCustom(nn.Module):
    """2D max pooling (no padding, floor mode — nn.MaxPool2d's defaults)."""

    def __init__(self, kernel_size, stride=None):
        super(MaxPool2dCustom, self).__init__()
        self.kernel_size = _pair(kernel_size)
        self.stride = _pair(stride) if stride is not None else self.kernel_size  # nn.MaxPool2d's default

    def forward(self, x):
        KH, KW = self.kernel_size
        SH, SW = self.stride
        windows = x.unfold(2, KH, SH).unfold(3, KW, SW)  # (N, C, OH, OW, KH, KW)
        return windows.amax(dim=(-2, -1))

In [ ]:
# Sanity check against PyTorch's built-ins: same weights in, same numbers out
torch.manual_seed(0)
for k, s, p in [(3, 1, 0), (3, 2, 1), (5, 3, 2)]:
    conv = Conv2dCustom(3, 4, k, stride=s, padding=p)
    x = torch.randn(2, 3, 11, 14)  # non-square on purpose
    expected = F.conv2d(x, conv.weight, conv.bias, stride=s, padding=p)
    assert torch.allclose(conv(x), expected, atol=1e-5), f"conv mismatch for kernel={k}, stride={s}, padding={p}"
for k, s in [(2, 2), (3, 2), (3, None)]:
    x = torch.randn(2, 3, 11, 14)
    expected = F.max_pool2d(x, k, stride=s)
    assert torch.equal(MaxPool2dCustom(k, stride=s)(x), expected), f"max-pool mismatch for kernel={k}, stride={s}"
print("Conv2dCustom and MaxPool2dCustom match F.conv2d and F.max_pool2d")

In [ ]:
# Define the CNN Model with the custom layers
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        self.conv1 = Conv2dCustom(3, 32, kernel_size=3, stride=1, padding=1)  # Output: 32x32x32
        self.conv2 = Conv2dCustom(32, 64, kernel_size=3, stride=1, padding=1)  # Output: 64x32x32
        self.pool = MaxPool2dCustom(kernel_size=2, stride=2)  # Output: 64x16x16
        self.fc1 = nn.Linear(64 * 16 * 16, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)  # Flatten
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
# Initialize the model, loss function, and optimizer
model = CNNModel()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop: the custom layers are several times slower than PyTorch's optimized
# kernels, so a few epochs are enough to see the model learn
epochs = 3
for epoch in range(epochs):
    for images, labels in train_loader:
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")

In [ ]:
# Evaluate on the test set
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("cnn-scratch", Conv2dCustom=Conv2dCustom, MaxPool2dCustom=MaxPool2dCustom)